# Metrics CERCA

In [1]:
import pandas as pd
import gender_guesser.detector as gender
from df2gspread import gspread2df as g2d

from tqdm import tqdm
tqdm.pandas()

In [2]:
df_whole = pd.read_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_segonaentrega.csv')

df = pd.read_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_OA_segonaentrega.csv')
df

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center
0,10.1038/s41598-022-20957-3,Marta Itarte,24.0,middle,False,7.199913e+07,ES,False,ICRA
1,10.1002/lno.11963,Anastasia Hiskia,55.0,middle,True,2.034740e+08,GR,False,ICRA
2,10.3389/fmicb.2021.678057,NaN,NaN,NaN,NaN,NaN,NaN,False,ICRA
3,10.5194/gmd-15-4597-2022,Orlane Anneville,39.0,middle,False,4.210089e+09,FR,False,ICRA
4,10.5194/gmd-15-4597-2022,Orlane Anneville,39.0,middle,False,7.090017e+07,FR,False,ICRA
...,...,...,...,...,...,...,...,...,...
8035,10.2166/9781789061154_0163,Jean‐Philippe Steyer,18.0,middle,False,4.210112e+09,FR,False,ICRA
8036,10.1021/acs.est.2c02896,Ivo Iavicoli,18.0,middle,False,7.126756e+07,IT,False,ICRA
8037,10.3390/w13172352,Eric D. van Hullebusch,18.0,middle,False,1.294672e+09,FR,False,ICRA
8038,10.3390/w13172352,Eric D. van Hullebusch,18.0,middle,False,2.047302e+08,FR,False,ICRA


In [3]:
df_whole.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
CRM     390
ICRA    543
dtype: int64

In [4]:
df[df.CERCA == True].drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
CRM     306
ICRA    480
dtype: int64

## Publications Number

In [5]:
print('The total percentage of publications analyzed is:', df.DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.9892818863879957


In [6]:
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
CRM     387
ICRA    536
dtype: int64

## % led publications

In [7]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.842443729903537


**Percentage computed with the retrieved OA-raw_affiliation data identified author and not the whole**

In [ ]:
df_cerca = df[df.CERCA == True]
df_led = df_cerca[(df_cerca.author_position == 'first') | (df_cerca.author_position == 'last') |(df_cerca.is_corresponding == True)]

df_led.drop_duplicates(['DOI', 'Center']).groupby('Center').size() / \
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
CRM     0.651163
ICRA    0.492537
dtype: float64

## \% publications with women from the centre as authors

In [9]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.842443729903537


**Percentage computed with the retrieved OA-raw_affiliation data identified author and not the whole**

In [10]:
gend = gender.Detector()

df_cerca = df[df.CERCA == True].dropna(subset = 'display_name') # TO DELETE NON FOUND AUTHORS

df_cerca['first_name'] = df_cerca['display_name'].str.split(' ').str[0]
df_cerca['gender'] = df_cerca.first_name.progress_apply(lambda x: gend.get_gender(x))

df_cerca.drop_duplicates('first_name').sort_values('first_name', ascending = False).to_csv('gender_check_segonaentrega.csv', index = False)
df_cerca

100%|██████████| 2751/2751 [00:00<00:00, 351580.80it/s]


,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center,first_name,gender
8,10.1038/s41598-022-20957-3,Josep Pueyo‐Ros,33.0,middle,False,2.514242e+08,ES,True,ICRA,Josep,male
36,10.5194/gmd-17-1-2024,Daniel Mercado‐Bettín,57.0,middle,False,4.210126e+09,US,True,ICRA,Daniel,male
37,10.5194/gmd-17-1-2024,Daniel Mercado‐Bettín,57.0,middle,False,4.387930e+09,NaN,True,ICRA,Daniel,male
129,10.1038/s43247-021-00192-w,Anna Freixa,38.0,middle,False,2.799563e+09,ES,True,ICRA,Anna,female
130,10.1038/s43247-021-00192-w,Anna Freixa,38.0,middle,False,2.514242e+08,ES,True,ICRA,Anna,female
...,...,...,...,...,...,...,...,...,...,...,...
7974,10.5194/hess-27-1361-2023,Rafael Marcé,15.0,last,True,2.514242e+08,ES,True,ICRA,Rafael,male
7975,10.5194/hess-27-1361-2023,Rafael Marcé,15.0,last,True,2.799563e+09,ES,True,ICRA,Rafael,male
8019,10.1016/j.scitotenv.2021.151925,Rafael Marcé,18.0,middle,False,2.514242e+08,ES,True,ICRA,Rafael,male
8020,10.1016/j.scitotenv.2021.151925,Rafael Marcé,18.0,middle,False,2.799563e+09,ES,True,ICRA,Rafael,male


**We manually revise the classifier**

In [11]:
gender_check = g2d.download('1J7uywXX7fsxjNbUSi24uQJiW5bTjbKHba1RjqdYBwQA', 'Gender', col_names = True, row_names = False)
df_cerca = df_cerca.merge(gender_check[['first_name', 'gender_check']], on='first_name', how='left')
df_cerca['gender'] = df_cerca.apply(lambda row: row['gender_check'] if row['gender_check'] != '' else row['gender'], axis = 1)
df_cerca

Not all requested scopes were granted by the authorization server, missing scopes https://spreadsheets.google.com/feeds, https://docs.google.com/feeds.


,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center,first_name,gender,gender_check
0,10.1038/s41598-022-20957-3,Josep Pueyo‐Ros,33.0,middle,False,2.514242e+08,ES,True,ICRA,Josep,male,
1,10.5194/gmd-17-1-2024,Daniel Mercado‐Bettín,57.0,middle,False,4.210126e+09,US,True,ICRA,Daniel,male,
2,10.5194/gmd-17-1-2024,Daniel Mercado‐Bettín,57.0,middle,False,4.387930e+09,NaN,True,ICRA,Daniel,male,
3,10.1038/s43247-021-00192-w,Anna Freixa,38.0,middle,False,2.799563e+09,ES,True,ICRA,Anna,female,
4,10.1038/s43247-021-00192-w,Anna Freixa,38.0,middle,False,2.514242e+08,ES,True,ICRA,Anna,female,
...,...,...,...,...,...,...,...,...,...,...,...,...
2746,10.5194/hess-27-1361-2023,Rafael Marcé,15.0,last,True,2.514242e+08,ES,True,ICRA,Rafael,male,
2747,10.5194/hess-27-1361-2023,Rafael Marcé,15.0,last,True,2.799563e+09,ES,True,ICRA,Rafael,male,
2748,10.1016/j.scitotenv.2021.151925,Rafael Marcé,18.0,middle,False,2.514242e+08,ES,True,ICRA,Rafael,male,
2749,10.1016/j.scitotenv.2021.151925,Rafael Marcé,18.0,middle,False,2.799563e+09,ES,True,ICRA,Rafael,male,


In [12]:
df_fem = df_cerca[df_cerca.gender.isin(['female'])]

df_fem.drop_duplicates(['DOI', 'Center']).groupby('Center').size() / \
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
CRM     0.162791
ICRA    0.341418
dtype: float64

## \% publications led by women from the centre as authors

In [13]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.842443729903537


**Percentage computed with the retrieved OA-raw_affiliation data identified author and not the whole**

In [14]:
df_fem_led = df_fem[(df_fem.author_position == 'first') | (df_fem.author_position == 'last') |(df_fem.is_corresponding == True) ]

df_fem_led.drop_duplicates(['DOI', 'Center']).groupby('Center').size() / \
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
CRM     0.105943
ICRA    0.203358
dtype: float64

## \% publications in collaboration with other CERCA centres

In [15]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.842443729903537


In [36]:
cerca_centers = {'CRM' : ['4210122226'], # NOT IN OA
                 'ICRA' : ['2799562678']}

df_cerca_af = pd.read_csv('../data/external/ToCheck - AffID.csv')

for institution in df_cerca.Center.unique():
    df_center = df[(df.Center == institution) & (df.institution_id != int(cerca_centers[institution][0]))]
    df_colab = df_center[df_center.institution_id.isin(df_cerca_af.OA_id)] 
    percentage = df_colab.DOI.nunique() / df_center.DOI.nunique()
    print(f"The percentage of publications in collaboration for {institution} is: {percentage:.2%}")

The percentage of publications in collaboration for ICRA is: 5.45%
The percentage of publications in collaboration for CRM is: 4.44%


In [17]:
# df_unique = df[['DOI', 'Center', 'CERCA']].drop_duplicates()
# df_cerca = df_unique[df_unique['CERCA'] == True]
# for institution in df_cerca.Center.unique():
#     dois_this = set(df_cerca.loc[df_cerca.Center == institution, 'DOI'])
#     dois_others = set(df_cerca.loc[df_cerca.Center != institution, 'DOI'])
#     collaborative_dois = dois_this & dois_others   
#     percentage = len(collaborative_dois) / len(dois_this) if dois_this else 0
#     print(f"The percentage of publications in collaboration for {institution} is: {percentage:.2%}")

## \% publications in collaboration with other local institutions

In [30]:
print('The total percentage of publications analyzed is:', df.DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.9892818863879957


In [32]:
cerca_centers = {'CRM' : ['4210122226'], # NOT IN OA
                 'ICRA' : ['2799562678']}

df_cerca_af = pd.read_csv('../data/external/ToCheck - AffID.csv')

for institution in df_cerca.Center.unique():
    df_center = df[(df.Center == institution) & (df.institution_id != int(cerca_centers[institution][0]))]
    df_cerca_colab = df_center[df_center.institution_id.isin(df_cerca_af.OA_id)] 
    df_not_cerca_colab = df_center[(~df_center.DOI.isin(df_cerca_colab.DOI)) & (df_center.COUNTRY_CODE == 'ES')]
    percentage = df_not_cerca_colab.DOI.nunique() / df_center.DOI.nunique()
    print(f"The percentage of publications in collaboration for {institution} is: {percentage:.2%}")

The percentage of publications in collaboration for ICRA is: 81.95%
The percentage of publications in collaboration for CRM is: 78.59%


In [33]:
# df_unique = df[['DOI', 'Center', 'CERCA', 'COUNTRY_CODE']].drop_duplicates()
# df_cerca = df_unique[df_unique['CERCA'] == True]
# df_spanish_non_cerca = df_unique[(df_unique['CERCA'] == False) & (df_unique['COUNTRY_CODE'] == 'ES')]
# for center in df_cerca['Center'].dropna().unique():
#     dois_center = set(df_cerca.loc[df_cerca['Center'] == center, 'DOI'])
#     dois_spanish_non_cerca = set(df_spanish_non_cerca['DOI'])
#     collaborative_dois = dois_center & dois_spanish_non_cerca
#     percentage = len(collaborative_dois) / len(dois_center)
#     print(f'The percentage of publications analyzed for {center} is: {percentage:.2%}')

## \% publications in collaboration with other international institutions

In [34]:
df_unique = df[['DOI', 'Center', 'CERCA', 'COUNTRY_CODE']].drop_duplicates()

df_cerca = df_unique[df_unique['CERCA'] == True]
df_international_non_cerca = df_unique[(df_unique['CERCA'] == False) & (df_unique['COUNTRY_CODE'] != 'ES')]

for center in df_cerca['Center'].dropna().unique():
    dois_center = set(df_cerca.loc[df_cerca['Center'] == center, 'DOI'])
    dois_international = set(df_international_non_cerca['DOI'])

    collaborative_dois = dois_center & dois_international
    
    percentage = len(collaborative_dois) / len(dois_center) if dois_center else 0
    print(f'The percentage of international collaborations for {center} is: {percentage:.2%}')

The percentage of international collaborations for ICRA is: 68.54%
The percentage of international collaborations for CRM is: 61.44%
